# E-Commerce Competitive Intelligence Multi-Agent System
## Capstone Assignment - 8-Week Production-Grade Project

**Course**: Agentic AI Bootcamp - Staff-Level System Design

---

## Assignment Overview

In this capstone project, you will build a **production-grade multi-agent competitive intelligence system** for e-commerce. This assignment guides you through implementing:

- Real-time competitor pricing analysis with AI
- Product feature comparison using semantic understanding
- Automated competitive marketing content generation
- Product knowledge graphs for relationship mapping
- Full observability with Langfuse tracing

### How to Use This Assignment

1. **Read the context** - Each section explains the business problem and concepts
2. **Follow TODO markers** - Complete each `# TODO:` with your implementation
3. **Use hints** - `HINT:` comments provide guidance without giving answers
4. **Check expected output** - Verify your implementation matches expected results
5. **Run validation cells** - Each week ends with a checkpoint to verify completion

### Grading Rubric

| Component | Points | Criteria |
|-----------|--------|----------|
| Week 1-2: Foundation | 15 | Environment, data processing |
| Week 3: Vector Store | 15 | Semantic search implementation |
| Week 4: Single Agents | 25 | Three working agents with tracing |
| Week 5: Orchestration | 20 | Multi-agent coordination |
| Week 6: Knowledge Graph | 15 | Graph construction and queries |
| Week 7-8: Integration | 10 | Full system with UI |

### 8-Week Curriculum

| Week | Module | Key Deliverables |
|------|--------|------------------|
| 1 | Environment & Data Foundation | Langfuse setup, catalog loading, EDA |
| 2 | Document Processing | Catalog normalization, feature extraction |
| 3 | Vector Store & Embeddings | ChromaDB, semantic search |
| 4 | Single Agent Design | Price, Catalog, Marketing agents |
| 5 | Multi-Agent Orchestration | Parallel analysis, aggregation |
| 6 | Knowledge Graphs | Product taxonomy, relationships |
| 7-8 | Production Deployment | Full integration, Gradio UI |

---

# WEEK 1: Environment Setup & Langfuse Initialization

---

## Learning Objectives

By the end of Week 1, you will be able to:

- [ ] Configure a production-ready development environment
- [ ] Initialize Langfuse observability from the first line of code
- [ ] Understand e-commerce product data structures
- [ ] Perform exploratory data analysis on product catalogs
- [ ] Create traced wrapper functions for API calls

## Why Observability First?

In competitive intelligence, every AI decision can impact business outcomes:
- A pricing recommendation could affect millions in revenue
- Marketing claims must be factually defensible
- Feature comparisons need accuracy for legal compliance

**Langfuse provides the audit trail** that makes AI decisions transparent and reviewable.

## 1.1 Package Installation

Install all required packages. Each serves a specific purpose in our architecture.

In [ ]:
# ============================================================================
# WEEK 1.1: PACKAGE INSTALLATION
# ============================================================================
# These packages are pre-configured - just run this cell

import subprocess, sys

_packages = ["openai==1.59.6", "langfuse==2.57.1", "langchain==0.3.14", "langchain-openai==0.2.14", "langchain-community==0.3.14", "langchain-core==0.3.29", "chromadb", "networkx", "pyvis", "python-docx", "openpyxl", "plotly", "seaborn", "pydantic>=2.0", "tenacity", "rich", "gradio", "python-dotenv", "numpy", "pandas", "matplotlib"]

print("Installing packages (this may take 1-2 minutes)...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + _packages,
    capture_output=True, text=True
)

if result.returncode != 0:
    for line in result.stderr.split("\n"):
        if line.strip() and "dependency resolver" not in line.lower() and "notice" not in line.lower():
            print(line)

# ---------- Verify EVERY import ----------
_verify = [
    ("openai", "openai"),
    ("langfuse", "langfuse"),
    ("langchain", "langchain"),
    ("langchain_openai", "langchain_openai"),
    ("chromadb", "chromadb"),
    ("networkx", "networkx"),
    ("pyvis", "pyvis.network"),
    ("docx", "docx"),
    ("openpyxl", "openpyxl"),
    ("plotly", "plotly"),
    ("seaborn", "seaborn"),
    ("pydantic", "pydantic"),
    ("tenacity", "tenacity"),
    ("rich", "rich"),
    ("gradio", "gradio"),
    ("dotenv", "dotenv"),
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("fastapi", "fastapi"),
    ("uvicorn", "uvicorn"),
]

_failed = []
for name, imp in _verify:
    try:
        __import__(imp)
    except ImportError:
        _failed.append(name)

if _failed:
    msg = f"FATAL: These packages failed to import: {', '.join(_failed)}\n"
    msg += "Try: Runtime > Restart runtime, then re-run this cell."
    raise ImportError(msg)

print("=" * 60)
print(f"All {len(_verify)} packages installed and verified!")
print("=" * 60)

In [ ]:
# ============================================================================
# CORE IMPORTS
# ============================================================================

import os
import sys
import json
import re
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional, Any, Tuple
from dataclasses import dataclass, field
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

print("Core imports completed successfully")

## 1.2 Environment Detection & API Configuration

**TODO**: Complete the environment detection and API key loading logic.

In [ ]:
# ============================================================================
# ENVIRONMENT DETECTION
# ============================================================================

# Detect if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

print(f"Environment Detection:")
print(f"  Running in Colab: {IN_COLAB}")
print(f"  Python version: {sys.version.split()[0]}")

In [ ]:
# ============================================================================
# API KEY CONFIGURATION & DATA INGESTION
# ============================================================================
# API keys from Colab Secrets or .env file
# Datasets fetched programmatically from GitHub

import subprocess

DATASET_REPO = "https://github.com/AI-Project-Lab/IK-pwc-agenticai-datasets.git"
DATASET_PROJECT = "ecommerce"

if IN_COLAB:
    print("Configuring for Google Colab environment...")

    # --- API Key Configuration ---
    try:
        from google.colab import userdata
        os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
        os.environ['LANGFUSE_SECRET_KEY'] = userdata.get('LANGFUSE_SECRET_KEY')
        os.environ['LANGFUSE_PUBLIC_KEY'] = userdata.get('LANGFUSE_PUBLIC_KEY')
        _host = userdata.get('LANGFUSE_HOST') or ''
        os.environ['LANGFUSE_HOST'] = _host if _host.startswith('http') else 'https://cloud.langfuse.com'
        print("API keys loaded from Colab Secrets")
    except Exception as e:
        print(f"Colab Secrets not available: {e}")
        import getpass
        os.environ['OPENAI_API_KEY'] = getpass.getpass('Enter OpenAI API Key: ')
        os.environ['LANGFUSE_SECRET_KEY'] = getpass.getpass('Enter Langfuse Secret Key: ')
        os.environ['LANGFUSE_PUBLIC_KEY'] = getpass.getpass('Enter Langfuse Public Key: ')
        os.environ['LANGFUSE_HOST'] = 'https://cloud.langfuse.com'

    # --- Dataset Ingestion from GitHub ---
    dataset_path = "/content/datasets"
    if not os.path.exists(f"{dataset_path}/{DATASET_PROJECT}"):
        print(f"\nCloning dataset from {DATASET_REPO}...")
        subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO, dataset_path],
                       check=True, capture_output=True)
        print("Dataset cloned successfully!")
    else:
        print("\nDataset already available.")
    DATA_DIR_BASE = f"{dataset_path}/{DATASET_PROJECT}"
else:
    print("Configuring for local environment...")
    from dotenv import load_dotenv
    load_dotenv()

    # --- Dataset Ingestion from GitHub ---
    datasets_parent = Path('.').resolve().parent.parent / 'datasets'
    if not (datasets_parent / DATASET_PROJECT).exists():
        print(f"\nCloning dataset from {DATASET_REPO}...")
        subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO, str(datasets_parent)],
                       check=True, capture_output=True)
        print("Dataset cloned successfully!")
    else:
        print("\nDataset already available locally.")
    DATA_DIR_BASE = str(datasets_parent / DATASET_PROJECT)


# Validate API keys
required_keys = ['OPENAI_API_KEY', 'LANGFUSE_SECRET_KEY', 'LANGFUSE_PUBLIC_KEY']
missing_keys = [key for key in required_keys if not os.environ.get(key)]
if missing_keys:
    raise EnvironmentError(f"Missing required API keys: {missing_keys}")

DATA_DIR = Path(DATA_DIR_BASE)
print(f"\nAll API keys validated successfully")
print(f"Data directory: {DATA_DIR}")
print(f"Directory exists: {DATA_DIR.exists()}")
if DATA_DIR.exists():
    file_count = sum(1 for _ in DATA_DIR.rglob('*') if _.is_file())
    print(f"Total files found: {file_count}")

## 1.3 Langfuse Initialization

Initialize Langfuse - the foundation of our observability strategy.

**TODO**: Initialize Langfuse client and verify the connection.

In [ ]:
# ============================================================================
# WEEK 1.3: LANGFUSE INITIALIZATION
# ============================================================================

from langfuse import Langfuse
from langfuse.callback import CallbackHandler as LangfuseCallbackHandler
from openai import OpenAI

# TODO: Initialize Langfuse client
# HINT: Langfuse() takes secret_key, public_key, and host parameters
langfuse = None  # Replace with Langfuse initialization

# TODO: Verify the connection using langfuse.auth_check()
# HINT: Wrap in try/except to handle connection failures gracefully

print("Langfuse initialization complete")

In [ ]:
# ============================================================================
# SESSION CONFIGURATION
# ============================================================================

# TODO: Create a unique session ID for this notebook run
# HINT: Include timestamp for uniqueness, e.g., f"ecommerce-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
SESSION_ID = None  # Replace with your session ID

# TODO: Initialize OpenAI client
openai_client = None  # Replace with OpenAI() initialization

print(f"\nSession Configuration:")
print(f"  Session ID: {SESSION_ID}")
print(f"  Langfuse Host: {os.environ.get('LANGFUSE_HOST')}")

## 1.4 Traced Wrapper Functions

Create wrapper functions that automatically trace all OpenAI API calls.

**TODO**: Implement traced embedding and completion functions.

In [ ]:
# ============================================================================
# TODO: TRACED EMBEDDING FUNCTION
# ============================================================================

def traced_embedding(
    text: str,
    trace_name: str = "embedding",
    metadata: Dict = None
) -> List[float]:
    """
    Generate embedding with full Langfuse tracing.

    TODO: Implement this function to:
    1. Create a Langfuse trace with the given name and session_id
    2. Create a generation span for the embedding call
    3. Call OpenAI embeddings API (model: text-embedding-3-small)
    4. Log the output and usage to the generation
    5. Return the embedding vector

    Args:
        text: Text to embed
        trace_name: Name for the trace
        metadata: Additional metadata to log

    Returns:
        List of embedding values

    HINTS:
    - Use langfuse.trace() to create a trace
    - Use trace.generation() to create a generation span
    - Use openai_client.embeddings.create() for the API call
    - Don't forget to call generation.end() with output and usage
    """
    # TODO: Implement traced embedding
    pass

print("traced_embedding() function defined (needs implementation)")

In [ ]:
# ============================================================================
# TODO: TRACED COMPLETION FUNCTION
# ============================================================================

def traced_completion(
    prompt: str,
    system: str = "",
    trace_name: str = "completion",
    model: str = "gpt-4o-mini",
    temperature: float = 0,
    metadata: Dict = None
) -> str:
    """
    Generate completion with full Langfuse tracing.

    TODO: Implement this function to:
    1. Create a Langfuse trace with metadata
    2. Build the messages list (system + user)
    3. Create a generation span
    4. Call OpenAI chat completions API
    5. Log usage (input_tokens, output_tokens, total_tokens)
    6. Return the response text

    Args:
        prompt: User prompt
        system: System message
        trace_name: Name for the trace
        model: Model to use
        temperature: Sampling temperature
        metadata: Additional metadata

    Returns:
        Model response text

    HINTS:
    - Build messages as: [{"role": "system", "content": ...}, {"role": "user", "content": ...}]
    - Use openai_client.chat.completions.create()
    - Response text is at: response.choices[0].message.content
    - Token counts are at: response.usage.prompt_tokens, completion_tokens, total_tokens
    """
    # TODO: Implement traced completion
    pass

print("traced_completion() function defined (needs implementation)")

In [ ]:
# ============================================================================
# TODO: LANGCHAIN CALLBACK HANDLER
# ============================================================================

def get_langfuse_handler(
    trace_name: str,
    tags: List[str] = None
) -> LangfuseCallbackHandler:
    """
    Get a LangChain callback handler for automatic tracing.

    TODO: Return a configured LangfuseCallbackHandler with:
    - secret_key, public_key, host from environment
    - session_id set to SESSION_ID
    - trace_name and tags as provided

    HINT: LangfuseCallbackHandler takes secret_key, public_key, host,
          session_id, trace_name, and tags parameters
    """
    # TODO: Implement callback handler factory
    pass

print("get_langfuse_handler() function defined (needs implementation)")

## 1.5 E-Commerce Product Catalogs

Load the product data. These catalogs represent two competing companies.

In [ ]:
# ============================================================================
# PRODUCT CATALOG - COMPANY X (Our Company)
# ============================================================================
# This data is provided - just run this cell

COMPANY_X_CATALOG = {
    "company": "Company X",
    "description": "Premium consumer electronics brand",
    "products": [
        {
            "category": "Wireless Headphones",
            "product_name": "Headphones X1",
            "price": 99.99,
            "currency": "USD",
            "features": ["Bluetooth 5.0", "Noise Cancelling", "20h Battery", "Foldable"],
            "discount": "10% off",
            "availability": "In Stock",
            "sku": "X1-HP-001"
        },
        {
            "category": "Wireless Headphones",
            "product_name": "Headphones X2 Pro",
            "price": 149.99,
            "currency": "USD",
            "features": ["Bluetooth 5.2", "Advanced ANC", "30h Battery", "Foldable", "USB-C", "Multipoint"],
            "discount": None,
            "availability": "In Stock",
            "sku": "X2-HP-002"
        },
        {
            "category": "Smart Watches",
            "product_name": "Watch X1",
            "price": 199.99,
            "currency": "USD",
            "features": ["Heart Rate", "GPS", "5 Day Battery", "Water Resistant", "Sleep Tracking"],
            "discount": "15% off",
            "availability": "In Stock",
            "sku": "X1-SW-001"
        },
        {
            "category": "Smart Watches",
            "product_name": "Watch X2 Ultra",
            "price": 349.99,
            "currency": "USD",
            "features": ["Heart Rate", "GPS", "ECG", "10 Day Battery", "Titanium", "Dive Mode"],
            "discount": None,
            "availability": "Limited Stock",
            "sku": "X2-SW-002"
        },
        {
            "category": "Portable Speakers",
            "product_name": "Speaker X1",
            "price": 79.99,
            "currency": "USD",
            "features": ["Bluetooth 5.0", "Waterproof", "12h Battery", "360 Sound"],
            "discount": None,
            "availability": "In Stock",
            "sku": "X1-SP-001"
        },
        {
            "category": "Portable Speakers",
            "product_name": "Speaker X2 Party",
            "price": 149.99,
            "currency": "USD",
            "features": ["Bluetooth 5.2", "Waterproof", "24h Battery", "LED Lights", "Mic Input"],
            "discount": "20% off",
            "availability": "In Stock",
            "sku": "X2-SP-002"
        }
    ]
}

print(f"Company X Catalog Loaded: {len(COMPANY_X_CATALOG['products'])} products")

In [ ]:
# ============================================================================
# PRODUCT CATALOG - COMPANY Y (Competitor)
# ============================================================================

COMPANY_Y_CATALOG = {
    "company": "Company Y",
    "description": "Value-focused consumer electronics",
    "products": [
        {
            "category": "Wireless Headphones",
            "product_name": "Headphones Z1",
            "price": 105.00,
            "currency": "USD",
            "features": ["Bluetooth 5.2", "Noise Cancelling", "25h Battery", "Quick Charge", "Foldable"],
            "discount": "5% off + Free Case",
            "availability": "In Stock",
            "sku": "Z1-HP-001"
        },
        {
            "category": "Wireless Headphones",
            "product_name": "Headphones Z2",
            "price": 115.00,
            "currency": "USD",
            "features": ["Bluetooth 5.2", "Advanced Noise Cancelling", "30h Battery", "Waterproof"],
            "discount": None,
            "availability": "In Stock",
            "sku": "Z2-HP-002"
        },
        {
            "category": "Smart Watches",
            "product_name": "Watch Z1",
            "price": 189.99,
            "currency": "USD",
            "features": ["Heart Rate", "GPS", "7 Day Battery", "Water Resistant", "Sleep Tracking", "SpO2"],
            "discount": "20% off",
            "availability": "In Stock",
            "sku": "Z1-SW-001"
        },
        {
            "category": "Smart Watches",
            "product_name": "Watch Z2 Sport",
            "price": 279.99,
            "currency": "USD",
            "features": ["Heart Rate", "GPS", "5 Day Battery", "100+ Workouts", "Voice Assistant"],
            "discount": "10% off",
            "availability": "In Stock",
            "sku": "Z2-SW-002"
        },
        {
            "category": "Portable Speakers",
            "product_name": "Speaker Z1",
            "price": 69.99,
            "currency": "USD",
            "features": ["Bluetooth 5.0", "Waterproof", "15h Battery", "Stereo Pairing"],
            "discount": "Free Shipping",
            "availability": "In Stock",
            "sku": "Z1-SP-001"
        },
        {
            "category": "Portable Speakers",
            "product_name": "Speaker Z2 Boom",
            "price": 129.99,
            "currency": "USD",
            "features": ["Bluetooth 5.2", "IP67 Rating", "20h Battery", "Bass Boost", "Daisy Chain"],
            "discount": None,
            "availability": "In Stock",
            "sku": "Z2-SP-002"
        }
    ]
}

print(f"Company Y Catalog Loaded: {len(COMPANY_Y_CATALOG['products'])} products")

## 1.6 Exploratory Data Analysis (EDA)

**TODO**: Build a DataFrame and analyze the product catalogs.

In [ ]:
# ============================================================================
# TODO: BUILD EDA DATAFRAME
# ============================================================================

import matplotlib.pyplot as plt
import seaborn as sns

# TODO: Create a list of dictionaries containing product data from both catalogs
# Each dictionary should have:
# - company: Company name
# - category: Product category
# - product: Product name
# - base_price: Original price
# - discount_pct: Discount percentage (parse from discount string, e.g., "10% off" -> 10)
# - effective_price: base_price * (1 - discount_pct/100)
# - feature_count: Number of features
# - has_discount: Boolean
#
# HINT: Use regex re.search(r'(\d+)%', discount_str) to extract percentage

eda_data = []

# TODO: Process COMPANY_X_CATALOG products
for product in COMPANY_X_CATALOG['products']:
    # Parse discount percentage
    discount_pct = 0
    # TODO: Extract discount percentage from product['discount']

    eda_data.append({
        # TODO: Fill in all required fields
    })

# TODO: Process COMPANY_Y_CATALOG products similarly

# Create DataFrame
eda_df = pd.DataFrame(eda_data)

print("EDA DataFrame created")
print(f"Shape: {eda_df.shape}")
print(f"Columns: {list(eda_df.columns)}")

In [ ]:
# ============================================================================
# TODO: EDA SUMMARY STATISTICS
# ============================================================================

print("PRODUCT CATALOG SUMMARY")
print("="*60)

# TODO: Display the following statistics:
# 1. Total number of products
# 2. Products by company (use value_counts())
# 3. Products by category
# 4. Average effective price by company (use groupby)
# 5. Number of products with discounts

# Expected output format:
# Total products: 12
# Products by Company:
#   Company X    6
#   Company Y    6
# etc.

In [ ]:
# ============================================================================
# TODO: EDA VISUALIZATION
# ============================================================================

# TODO: Create a 2x2 subplot figure showing:
# 1. Effective price comparison by product (bar chart)
# 2. Average price by category and company (grouped bar chart)
# 3. Average feature count by company (bar chart)
# 4. Scatter plot of price vs feature count, colored by company
#
# HINTS:
# - Use fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# - Company colors: {'Company X': '#3498db', 'Company Y': '#e74c3c'}
# - Save the figure as 'ecommerce_eda.png'

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# TODO: Implement visualizations

plt.tight_layout()
plt.savefig('ecommerce_eda.png', dpi=150, bbox_inches='tight')
plt.show()

print("EDA visualization saved to ecommerce_eda.png")

## Week 1 Checkpoint

Run this cell to verify your Week 1 implementation.

In [ ]:
# ============================================================================
# WEEK 1 CHECKPOINT - VALIDATION
# ============================================================================

print("WEEK 1 CHECKPOINT - Verification")
print("="*60)

checks = [
    ("IN_COLAB is defined", IN_COLAB is not None),
    ("Langfuse initialized", langfuse is not None),
    ("SESSION_ID created", SESSION_ID is not None and len(str(SESSION_ID)) > 0),
    ("OpenAI client ready", openai_client is not None),
    ("traced_embedding() implemented", traced_embedding is not None and traced_embedding.__code__.co_code != b'd\x00S\x00'),
    ("traced_completion() implemented", traced_completion is not None and traced_completion.__code__.co_code != b'd\x00S\x00'),
    ("get_langfuse_handler() implemented", get_langfuse_handler is not None),
    ("Company X catalog loaded", len(COMPANY_X_CATALOG['products']) == 6),
    ("Company Y catalog loaded", len(COMPANY_Y_CATALOG['products']) == 6),
    ("EDA DataFrame created", 'eda_df' in dir() and len(eda_df) == 12),
]

passed = 0
for check_name, check_result in checks:
    status = "PASS" if check_result else "FAIL"
    icon = "[OK]" if check_result else "[X]"
    print(f"  {icon} {check_name}: {status}")
    if check_result:
        passed += 1

print("\n" + "="*60)
print(f"Week 1 Score: {passed}/{len(checks)} checks passed")
if passed == len(checks):
    print("Week 1 Complete! Proceed to Week 2.")
else:
    print("Review failed checks before continuing.")